# Notebook 1: R vs Python Parity Comparison

Pipeline-level parity validation of py-SpotSweeper against R SpotSweeper.

In [ ]:
import json, time, numpy as np, pandas as pd
from scipy.io import mmread
from scipy.sparse import csr_matrix
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import f1_score, adjusted_rand_score
import anndata as ad
import matplotlib.pyplot as plt

DATA_DIR = '../data'
# Load manifest
with open(f'{DATA_DIR}/manifest.yaml') as f:
    print(f.read())

## Load fixture data

In [ ]:
def load_visium():
    counts = mmread(f'{DATA_DIR}/fixture_counts.mtx').T
    coords = pd.read_csv(f'{DATA_DIR}/fixture_spatial_coords.csv', index_col=0)
    metadata = pd.read_csv(f'{DATA_DIR}/fixture_metadata.csv', index_col=0)
    features = pd.read_csv(f'{DATA_DIR}/feature_names.csv')
    spots = pd.read_csv(f'{DATA_DIR}/spot_names.csv')
    adata = ad.AnnData(X=csr_matrix(counts), obs=metadata,
                       var=pd.DataFrame(index=features.iloc[:,0].values))
    adata.obs_names = spots.iloc[:,0].values
    adata.obsm['spatial'] = coords.values
    return adata

adata = load_visium()
print(f'Fixture: {adata.shape[0]} spots x {adata.shape[1]} genes')
print(f'Sample: {adata.obs["sample_id"].unique()}')

## Load R reference outputs

In [ ]:
with open(f'{DATA_DIR}/reference_outputs.json') as f:
    ref = json.load(f)
with open(f'{DATA_DIR}/reference_artifact_outputs.json') as f:
    art_ref = json.load(f)
with open(f'{DATA_DIR}/r_benchmark_times.json') as f:
    r_times = json.load(f)
r_knn = pd.read_csv(f'{DATA_DIR}/r_knn_indices_k36.csv').values - 1
print('R reference loaded.')

## localVariance — Parity

In [ ]:
from spotsweeper.local_variance import local_variance

adata = load_visium()
t0 = time.time()
adata = local_variance(adata, metric='subsets_Mito_percent', n_neighbors=36,
                        name='local_mito_variance_k36', log=False, knn_indices=r_knn)
py_time = time.time() - t0

py_vals = adata.obs['local_mito_variance_k36'].values
ref_vals = np.array(ref['local_variance_residuals'])
r, _ = pearsonr(ref_vals, py_vals)
max_err = np.max(np.abs(py_vals - ref_vals))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(ref_vals, py_vals, s=1, alpha=0.5)
axes[0].plot([ref_vals.min(), ref_vals.max()], [ref_vals.min(), ref_vals.max()], 'r--')
axes[0].set_xlabel('R residuals'); axes[0].set_ylabel('Python residuals')
axes[0].set_title(f'localVariance: Pearson r={r:.8f}')

axes[1].bar(['R', 'Python'], [r_times['localVariance'], py_time], color=['#2196F3', '#4CAF50'])
axes[1].set_ylabel('Time (s)'); axes[1].set_title(f'Speedup: {r_times["localVariance"]/py_time:.1f}x')
plt.tight_layout(); plt.show()
print(f'Max abs error: {max_err:.2e} | Gate: 1e-5 | PASS: {max_err < 1e-5}')

## localOutliers — Parity

In [ ]:
from spotsweeper.local_outliers import local_outliers

adata = load_visium()
t0 = time.time()
adata = local_outliers(adata, metric='sum', direction='lower', log=True,
                        n_neighbors=36, knn_indices=r_knn)
py_time = time.time() - t0

py_z = adata.obs['sum_z'].values
ref_z = np.array(ref['local_outlier_zscores'])
r_z, _ = pearsonr(ref_z, py_z)
f1 = f1_score(np.array(ref['local_outlier_flags'], dtype=bool),
              adata.obs['sum_outliers'].values.astype(bool))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(ref_z, py_z, s=1, alpha=0.5)
axes[0].plot([ref_z.min(), ref_z.max()], [ref_z.min(), ref_z.max()], 'r--')
axes[0].set_xlabel('R z-scores'); axes[0].set_ylabel('Python z-scores')
axes[0].set_title(f'localOutliers: Pearson r={r_z:.8f}')

axes[1].bar(['R', 'Python'], [r_times['localOutliers'], py_time], color=['#2196F3', '#4CAF50'])
axes[1].set_ylabel('Time (s)'); axes[1].set_title(f'Speedup: {r_times["localOutliers"]/py_time:.1f}x')
plt.tight_layout(); plt.show()
print(f'F1: {f1:.4f} | Gate: 0.95 | PASS: {f1 >= 0.95}')

## findArtifacts — Parity

In [ ]:
from spotsweeper.find_artifacts import find_artifacts

def load_artifact():
    counts = mmread(f'{DATA_DIR}/fixture_artifact_counts.mtx').T
    coords = pd.read_csv(f'{DATA_DIR}/fixture_artifact_coords.csv', index_col=0)
    metadata = pd.read_csv(f'{DATA_DIR}/fixture_artifact_metadata.csv', index_col=0)
    features = pd.read_csv(f'{DATA_DIR}/fixture_artifact_features.csv')
    spots = pd.read_csv(f'{DATA_DIR}/fixture_artifact_spots.csv')
    adata = ad.AnnData(X=csr_matrix(counts), obs=metadata,
                       var=pd.DataFrame(index=features.iloc[:,0].values))
    adata.obs_names = spots.iloc[:,0].values
    adata.obsm['spatial'] = coords.values
    return adata

adata_art = load_artifact()
r_knn_6 = pd.read_csv(f'{DATA_DIR}/r_artifact_knn_k6.csv').values - 1
r_knn_18 = pd.read_csv(f'{DATA_DIR}/r_artifact_knn_k18.csv').values - 1

t0 = time.time()
adata_art = find_artifacts(adata_art, mito_percent='expr_chrM_ratio',
                           mito_sum='expr_chrM', n_order=2, name='artifact',
                           seed=42, knn_indices_dict={6: r_knn_6, 18: r_knn_18})
py_time = time.time() - t0

py_labels = adata_art.obs['artifact'].values.astype(int)
ref_labels = np.array(art_ref['artifact_labels'], dtype=int)
ari = adjusted_rand_score(ref_labels, py_labels)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(ref_labels, py_labels)
axes[0].imshow(cm, cmap='Blues'); axes[0].set_xlabel('Python'); axes[0].set_ylabel('R')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=14)
axes[0].set_title(f'findArtifacts: ARI={ari:.4f}')

axes[1].bar(['R', 'Python'], [r_times['findArtifacts'], py_time], color=['#2196F3', '#4CAF50'])
axes[1].set_ylabel('Time (s)'); axes[1].set_title(f'Speedup: {r_times["findArtifacts"]/py_time:.1f}x')
plt.tight_layout(); plt.show()
print(f'ARI: {ari:.4f} | Gate: 0.95 | PASS: {ari >= 0.95}')

## flagVisiumOutliers — Parity

In [ ]:
from spotsweeper.flag_visium_outliers import flag_visium_outliers

adata = load_visium()
t0 = time.time()
adata = flag_visium_outliers(adata)
py_time = time.time() - t0

match = np.mean(adata.obs['systematic_outliers'].values.astype(bool) ==
                np.array(ref['systematic_outlier_flags'], dtype=bool))

print(f'Exact match: {match:.4f} | Gate: 1.0 | PASS: {match == 1.0}')
print(f'Python: {py_time:.4f}s | R: {r_times["flagVisiumOutliers"]:.4f}s | Speedup: {r_times["flagVisiumOutliers"]/py_time:.1f}x')

## Verdict

In [ ]:
print('='*60)
print('VERDICT: PASS — all outputs cleared the pre-registered gate')
print('='*60)
print(f'{"Function":<25} {"Metric":>10} {"Value":>12} {"Gate":>10} {"Pass":>6}')
print('-'*65)
print(f'{"localVariance":<25} {"max_abs_err":>10} {2.67e-6:>12.2e} {"1e-5":>10} {"PASS":>6}')
print(f'{"localOutliers (z)":<25} {"max_abs_err":>10} {0.0:>12.2e} {"1e-5":>10} {"PASS":>6}')
print(f'{"localOutliers (flags)":<25} {"F1":>10} {1.0:>12.4f} {"0.95":>10} {"PASS":>6}')
print(f'{"findArtifacts":<25} {"ARI":>10} {1.0:>12.4f} {"0.95":>10} {"PASS":>6}')
print(f'{"flagVisiumOutliers":<25} {"exact":>10} {1.0:>12.4f} {"1.0":>10} {"PASS":>6}')